# 🐍 DuckDB Python Basics

This notebook covers the fundamentals of using DuckDB with Python.

## Table of Contents
1. [Installing & Importing DuckDB](#1-installing--importing-duckdb)
2. [Connecting to DuckDB](#2-connecting-to-duckdb)
3. [Executing Queries](#3-executing-queries)
4. [Fetching Results](#4-fetching-results)
5. [Pandas Integration](#5-pandas-integration)
6. [Working with CSV Files](#6-working-with-csv-files)
7. [CRUD Operations](#7-crud-operations)
8. [Best Practices](#8-best-practices)

## 1. Installing & Importing DuckDB

DuckDB can be installed via pip. It has no external dependencies!

In [1]:
# Install DuckDB (uncomment if needed)
# !pip install duckdb pandas

# Import libraries
import duckdb
import pandas as pd

# Check version
print(f"DuckDB version: {duckdb.__version__}")

DuckDB version: 1.4.2


## 2. Connecting to DuckDB

DuckDB supports two connection modes:
- **In-memory**: Data exists only during the session (default)
- **Persistent**: Data is saved to a file

In [2]:
# Method 1: In-memory database (default)
conn = duckdb.connect()
print("In-memory connection created!")

# Method 2: In-memory with explicit syntax
conn_memory = duckdb.connect(':memory:')
print("Explicit in-memory connection created!")

# Method 3: Persistent database (saved to file)
conn_persistent = duckdb.connect('my_database.db')
print("Persistent database 'my_database.db' created!")

# Always close connections when done
conn_persistent.close()

In-memory connection created!
Explicit in-memory connection created!
Persistent database 'my_database.db' created!


In [3]:
# Best practice: Use context manager for automatic cleanup
with duckdb.connect() as conn:
    result = conn.execute("SELECT 'Hello from DuckDB!' as greeting").fetchone()
    print(result[0])
# Connection automatically closed here

Hello from DuckDB!


## 3. Executing Queries

DuckDB provides multiple ways to execute SQL queries.

In [4]:
# Create a connection for the rest of this notebook
conn = duckdb.connect()

# Method 1: Using connection.execute()
conn.execute("CREATE TABLE users (id INTEGER, name VARCHAR, age INTEGER)")
conn.execute("INSERT INTO users VALUES (1, 'Alice', 30), (2, 'Bob', 25), (3, 'Charlie', 35)")
print("Table created and data inserted!")

# Method 2: Using connection.sql() - returns a relation
result = conn.sql("SELECT * FROM users")
print(result)

Table created and data inserted!
┌───────┬─────────┬───────┐
│  id   │  name   │  age  │
│ int32 │ varchar │ int32 │
├───────┼─────────┼───────┤
│     1 │ Alice   │    30 │
│     2 │ Bob     │    25 │
│     3 │ Charlie │    35 │
└───────┴─────────┴───────┘



In [5]:
# Method 3: Using duckdb.query() - uses default in-memory connection
result = duckdb.query("SELECT 42 as answer, 'DuckDB' as name")
print(result)

# Method 4: Parameterized queries (prevent SQL injection!)
name = "Alice"
age = 30
result = conn.execute("SELECT * FROM users WHERE name = ? AND age = ?", [name, age])
print(result.fetchall())

┌────────┬─────────┐
│ answer │  name   │
│ int32  │ varchar │
├────────┼─────────┤
│     42 │ DuckDB  │
└────────┴─────────┘

[(1, 'Alice', 30)]


## 4. Fetching Results

DuckDB offers multiple ways to retrieve query results.

In [6]:
# fetchone() - Get single row as tuple
result = conn.execute("SELECT * FROM users WHERE id = 1")
row = result.fetchone()
print(f"fetchone(): {row}")
print(f"Type: {type(row)}")

# fetchall() - Get all rows as list of tuples
result = conn.execute("SELECT * FROM users")
rows = result.fetchall()
print(f"\nfetchall(): {rows}")

# fetchmany(n) - Get n rows
result = conn.execute("SELECT * FROM users")
rows = result.fetchmany(2)
print(f"\nfetchmany(2): {rows}")

fetchone(): (1, 'Alice', 30)
Type: <class 'tuple'>

fetchall(): [(1, 'Alice', 30), (2, 'Bob', 25), (3, 'Charlie', 35)]

fetchmany(2): [(1, 'Alice', 30), (2, 'Bob', 25)]


In [7]:
# df() - Get results as pandas DataFrame (most common!)
df = conn.execute("SELECT * FROM users").df()
print("As DataFrame:")
print(df)
print(f"\nType: {type(df)}")

# fetchnumpy() - Get as dictionary of numpy arrays
result = conn.execute("SELECT * FROM users").fetchnumpy()
print(f"\nAs NumPy dict: {result}")

As DataFrame:
   id     name  age
0   1    Alice   30
1   2      Bob   25
2   3  Charlie   35

Type: <class 'pandas.core.frame.DataFrame'>

As NumPy dict: {'id': array([1, 2, 3], dtype=int32), 'name': array(['Alice', 'Bob', 'Charlie'], dtype=object), 'age': array([30, 25, 35], dtype=int32)}


## 5. Pandas Integration

DuckDB has excellent pandas integration - you can query DataFrames directly!

In [8]:
# Create a pandas DataFrame
sales_df = pd.DataFrame({
    'product': ['Widget', 'Gadget', 'Widget', 'Gadget', 'Widget'],
    'quantity': [10, 5, 8, 12, 3],
    'price': [9.99, 19.99, 9.99, 19.99, 9.99],
    'date': pd.to_datetime(['2024-01-01', '2024-01-01', '2024-01-02', '2024-01-02', '2024-01-03'])
})

print("Original DataFrame:")
print(sales_df)

Original DataFrame:
  product  quantity  price       date
0  Widget        10   9.99 2024-01-01
1  Gadget         5  19.99 2024-01-01
2  Widget         8   9.99 2024-01-02
3  Gadget        12  19.99 2024-01-02
4  Widget         3   9.99 2024-01-03


In [9]:
# Query the DataFrame directly with SQL!
# DuckDB automatically recognizes Python variables

result = duckdb.query("""
    SELECT 
        product,
        SUM(quantity) as total_qty,
        SUM(quantity * price) as total_revenue
    FROM sales_df
    GROUP BY product
    ORDER BY total_revenue DESC
""").df()

print("Aggregated with SQL:")
print(result)

Aggregated with SQL:
  product  total_qty  total_revenue
0  Gadget       17.0         339.83
1  Widget       21.0         209.79


In [10]:
# Join multiple DataFrames
products_df = pd.DataFrame({
    'product': ['Widget', 'Gadget'],
    'category': ['Tools', 'Electronics']
})

# Join using SQL
joined = duckdb.query("""
    SELECT s.*, p.category
    FROM sales_df s
    JOIN products_df p ON s.product = p.product
""").df()

print("Joined result:")
print(joined)

Joined result:
  product  quantity  price       date     category
0  Widget        10   9.99 2024-01-01        Tools
1  Gadget         5  19.99 2024-01-01  Electronics
2  Widget         8   9.99 2024-01-02        Tools
3  Gadget        12  19.99 2024-01-02  Electronics
4  Widget         3   9.99 2024-01-03        Tools


## 6. Working with CSV Files

DuckDB can read CSV files directly without loading them into memory first!

In [11]:
# Read CSV file directly
states_df = conn.execute("""
    SELECT * FROM read_csv_auto('../sample_data/states.csv')
    LIMIT 10
""").df()

print("States from CSV:")
print(states_df)

States from CSV:
     id        name  country_id country_code country_name state_code  type  \
0  3901  Badakhshan           1           AF  Afghanistan        BDS  None   
1  3871     Badghis           1           AF  Afghanistan        BDG  None   
2  3875     Baghlan           1           AF  Afghanistan        BGL  None   
3  3884       Balkh           1           AF  Afghanistan        BAL  None   
4  3872      Bamyan           1           AF  Afghanistan        BAM  None   
5  3892    Daykundi           1           AF  Afghanistan        DAY  None   
6  3899       Farah           1           AF  Afghanistan        FRA  None   
7  3889      Faryab           1           AF  Afghanistan        FYB  None   
8  3870      Ghazni           1           AF  Afghanistan        GHA  None   
9  3888        Ghōr           1           AF  Afghanistan        GHO  None   

    latitude  longitude  
0  36.734772  70.811995  
1  35.167134  63.769538  
2  36.178903  68.745306  
3  36.755060  66.897

In [12]:
# Query CSV with aggregation
result = conn.execute("""
    SELECT 
        COUNT(*) as total_products,
        AVG(ListPrice) as avg_price,
        MAX(ListPrice) as max_price
    FROM read_csv_auto('../sample_data/AW_CSV/Production.Product.csv')
    WHERE ListPrice > 0
""").df()

print("Product Statistics:")
print(result)

Product Statistics:
   total_products   avg_price  max_price
0             304  727.262467    3578.27


In [13]:
# Read multiple CSV files with glob pattern
# This reads all Person.*.csv files at once
result = conn.execute("""
    SELECT filename, COUNT(*) as rows
    FROM read_csv_auto('../sample_data/AW_CSV/Person.*.csv', filename=true)
    GROUP BY filename
""").df()

print("Rows per Person file:")
print(result)

Rows per Person file:
                                   filename   rows
0  ../sample_data/AW_CSV/Person.Address.csv  19614


## 7. CRUD Operations

Create, Read, Update, Delete operations in DuckDB.

In [18]:
# CREATE - Create a new table (OR REPLACE for re-running)
conn.execute("""
    CREATE OR REPLACE TABLE products (
        id INTEGER PRIMARY KEY,
        name VARCHAR NOT NULL,
        price DECIMAL(10,2),
        stock INTEGER DEFAULT 0,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")
print("Table 'products' created!")

# Show table schema
conn.execute("DESCRIBE products").df()

Table 'products' created!


,column_name,column_type,null,key,default,extra
0,id,INTEGER,NO,PRI,None,None
1,name,VARCHAR,NO,None,None,None
2,price,"DECIMAL(10,2)",YES,None,None,None
3,stock,INTEGER,YES,None,0,None
4,created_at,TIMESTAMP,YES,None,CURRENT_TIMESTAMP,None


In [19]:
# INSERT - Add data
# Single row
conn.execute("INSERT INTO products (id, name, price, stock) VALUES (1, 'Laptop', 999.99, 50)")

# Multiple rows
conn.execute("""
    INSERT INTO products (id, name, price, stock) VALUES 
    (2, 'Mouse', 29.99, 200),
    (3, 'Keyboard', 79.99, 150),
    (4, 'Monitor', 399.99, 30)
""")

# Insert from DataFrame (specify columns to match!)
new_products = pd.DataFrame({
    'id': [5, 6],
    'name': ['Webcam', 'Headset'],
    'price': [89.99, 149.99],
    'stock': [75, 60]
})
conn.execute("INSERT INTO products (id, name, price, stock) SELECT * FROM new_products")

print("Data inserted!")
conn.execute("SELECT * FROM products").df()

Data inserted!


,id,name,price,stock,created_at
0,1,Laptop,999.99,50,2025-11-26 22:59:55.606627
1,2,Mouse,29.99,200,2025-11-26 22:59:55.607129
2,3,Keyboard,79.99,150,2025-11-26 22:59:55.607129
3,4,Monitor,399.99,30,2025-11-26 22:59:55.607129
4,5,Webcam,89.99,75,2025-11-26 22:59:55.608258
5,6,Headset,149.99,60,2025-11-26 22:59:55.608258


In [20]:
# UPDATE - Modify existing data
conn.execute("UPDATE products SET price = price * 0.9 WHERE stock < 50")
print("Prices reduced for low-stock items!")

# DELETE - Remove data
conn.execute("DELETE FROM products WHERE id = 6")
print("Product deleted!")

# View updated data
conn.execute("SELECT * FROM products ORDER BY id").df()

Prices reduced for low-stock items!
Product deleted!


,id,name,price,stock,created_at
0,1,Laptop,999.99,50,2025-11-26 22:59:55.606627
1,2,Mouse,29.99,200,2025-11-26 22:59:55.607129
2,3,Keyboard,79.99,150,2025-11-26 22:59:55.607129
3,4,Monitor,359.99,30,2025-11-26 22:59:55.607129
4,5,Webcam,89.99,75,2025-11-26 22:59:55.608258


## 8. Best Practices

Tips for working efficiently with DuckDB in Python.

In [21]:
# 1. Use context managers for automatic cleanup
with duckdb.connect() as temp_conn:
    result = temp_conn.execute("SELECT 1 as test").fetchone()
    print(f"Context manager result: {result}")

# 2. Use .df() for pandas integration
df = conn.execute("SELECT * FROM products").df()

# 3. Use parameterized queries to prevent SQL injection
product_id = 1
result = conn.execute("SELECT * FROM products WHERE id = ?", [product_id]).df()

# 4. Query files directly without loading into tables
# This is faster for one-off queries!
result = conn.execute("SELECT COUNT(*) FROM '../sample_data/states.csv'").fetchone()
print(f"Direct file query: {result[0]} rows")

Context manager result: (1,)
Direct file query: 5077 rows


In [22]:
# 5. Export results to files
conn.execute("""
    COPY (SELECT * FROM products WHERE stock > 50) 
    TO '../exports/high_stock_products.csv' (HEADER, DELIMITER ',')
""")
print("Data exported to CSV!")

# 6. Clean up: Close connections when done
conn.close()
print("Connection closed!")

# Clean up temporary database file
import os
if os.path.exists('my_database.db'):
    os.remove('my_database.db')
    print("Temporary database removed!")

Data exported to CSV!
Connection closed!
Temporary database removed!


## 🎯 Summary

| Task | Method |
|------|--------|
| Connect | `duckdb.connect()` or `duckdb.connect('file.db')` |
| Execute SQL | `conn.execute(sql)` or `conn.sql(sql)` |
| Get DataFrame | `.df()` |
| Get tuples | `.fetchall()` or `.fetchone()` |
| Query DataFrame | `duckdb.query("SELECT * FROM df_name")` |
| Read CSV | `read_csv_auto('file.csv')` |
| Parameterized query | `conn.execute(sql, [params])` |